In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 19 samples and 12146 columns.
First few rows of the data:


,Time between relapse and death (months),Death,Patient ID,Group*,Age at the surgery (years),Sexa,Tumor size (cm)b (measured after surgery),T,N,Number of nodes analyzed,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,NaN,1,S1,ST,74,F,3.0,4,1,25,...,3.092221,2.639923,3.936343,3.672092,2.374791,2.476992,4.737094,4.382828,3.210585,3.485605
1,1.31,1,S2,ST,71,M,3.5,3,0,12,...,1.888021,1.028991,2.070466,2.147473,0.970617,1.101873,3.036564,2.897909,1.525291,1.830348
2,NaN,1,S3,ST,72,F,3.2,3,0,7,...,1.264847,0.526752,1.706065,1.496742,0.740946,0.639573,2.312651,2.346399,1.197374,1.629780
3,2.89,1,S4,ST,73,M,4.0,3,1,7,...,2.662550,1.882318,3.190156,3.233296,1.591031,1.953116,3.889154,3.760724,2.518112,3.048301
4,6.11,1,S5,ST,54,F,3.2,4,1,12,...,2.987803,1.997387,3.235144,2.978783,1.611226,1.941454,3.958125,3.768537,2.649038,2.863672


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Skipping stats for DFS: columns not in preprocessed table ('Time between surgery and relapse (months) (DFS)c', 'unknown').
Number of samples with incomplete data for Time between relapse and death (months) or Death: 9
Number of samples with complete data for both variables: 10
Number of censored samples (event_var == 0.0): 0
Number of events (event_var == 1.0): 10
Censored ratio (among complete): 0 / 10 = 0.000 (0.0%)
Skipping stats for OS: columns not in preprocessed table ('unknown', 'Death').
Updated stats for 1 of 3 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE150043/description.json
